# SolarNL — Opdracht 2: invloed van weersomstandigheden op zonnepanelen

**HvA Energielab / AMOLF · Technische Natuurkunde · 2026–2027**

Dit notebook onderzoekt hoe weersomstandigheden samenhangen met de paneeltemperatuur en de elektrische prestaties van acht zonnepanelen. Het is een vervolg op `SolarNL_EDA.ipynb`.

**Onderzoeksvraag:** Welke invloed hebben verschillende weersomstandigheden op de elektrische prestaties van de acht zonnepanelen, hoe beïnvloeden deze omstandigheden de paneeltemperatuur en welk effect heeft deze temperatuur op de elektrische prestaties?

**Belangrijke beperkingen:** de huidige export beslaat slechts enkele dagen; hiermee kan geen meerjarige degradatie worden aangetoond. Identieke elektrische waarden tussen sommige panelen, ontbrekende meetmomenten en de onbekende betekenis van `CMP-10` vereisen voorzichtigheid.

**Privacy:** de brondata blijft lokaal. Publiceer geen ruwe data of individuele meetrecords zonder toestemming van de opdrachtgevers.

## 1. Bibliotheken en bronbestand

Het notebook gebruikt de bestaande CSV in dezelfde map. Er worden geen gegevens verwijderd omdat andere variabelen ontbreken. De InfluxDB-export kan meerdere tabellen met herhaalde kopregels bevatten.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from io import StringIO
from IPython.display import display

BESTAND = Path("Datasheet_query_(10).csv")
if not BESTAND.exists():
    raise FileNotFoundError(f"CSV niet gevonden: {BESTAND.resolve()}")

with BESTAND.open("r", encoding="utf-8-sig") as f:
    regels = f.readlines()

tabellen, huidige = [], []
for regel in regels:
    if regel.startswith("#datatype"):
        if huidige:
            tabellen.append(huidige)
        huidige = []
    elif regel.startswith("#") or not regel.strip():
        continue
    else:
        huidige.append(regel)
if huidige:
    tabellen.append(huidige)

frames = []
for tabel in tabellen:
    frame = pd.read_csv(StringIO("".join(tabel)), sep=";")
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed")]
    frame = frame.drop(columns=["result", "table", "_start", "_stop"], errors="ignore")
    frame = frame.dropna(how="all")
    if "_time" in frame.columns:
        frame["_time"] = pd.to_datetime(frame["_time"], utc=True, errors="coerce")
    frames.append(frame)

solar_frames = [f for f in frames if "impp" in f.columns]
weather_frames = [f for f in frames if "Temperature-PVMET500" in f.columns]
if not solar_frames or not weather_frames:
    raise ValueError("Solar- of weerdata niet gevonden. Controleer de CSV-structuur.")

df_solar_raw = pd.concat(solar_frames, ignore_index=True)
df_weather = pd.concat(weather_frames, ignore_index=True)
print("Zonnepaneelrijen:", len(df_solar_raw))
print("Weerrijen:", len(df_weather))

## 2. Voorbereiden zonder te streng te filteren

De acht Pt100-kolommen worden samengevoegd. **Alleen een ontbrekende waarde van de variabele die je onderzoekt wordt per analyse uitgesloten.** Dat voorkomt onnodig verlies van temperatuurmetingen.

In [ ]:
df_solar = df_solar_raw.copy()
pt_cols = [c for c in df_solar.columns if c.startswith("Pt100") and c != "Pt100"]
if pt_cols:
    pt = df_solar[pt_cols].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]
    if "Pt100" in df_solar.columns:
        df_solar["Pt100"] = pd.to_numeric(df_solar["Pt100"], errors="coerce").combine_first(pt)
    else:
        df_solar["Pt100"] = pt
    df_solar = df_solar.drop(columns=pt_cols)

variabelen = ["Pt100", "ff", "impp", "isc", "vmpp", "voc", "CMP-10", "Ref"]
for col in variabelen:
    if col in df_solar.columns:
        df_solar[col] = pd.to_numeric(df_solar[col], errors="coerce")
for col in ["Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]:
    df_weather[col] = pd.to_numeric(df_weather[col], errors="coerce")

df_solar = df_solar.dropna(subset=["_time", "label"]).sort_values(["label", "_time"]).copy()
df_weather = df_weather.dropna(subset=["_time"]).sort_values("_time").copy()

display(df_solar.groupby("label")[["Pt100", "ff", "impp", "isc", "vmpp", "voc"]].count())
print("Beschikbare weersensorwaarden:")
display(df_weather[["Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]].count())

## 3. Synchronisatie met het weerstation

Weerwaarden worden via de **dichtstbijzijnde meting binnen twee minuten** aan zonnepaneelmetingen gekoppeld. We gebruiken UTC om tijdzone- en wintertijdproblemen te vermijden. Een ontbrekende match blijft ontbrekend.

In [ ]:
weer = (df_weather[["_time", "Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]]
        .drop_duplicates(subset=["_time"])
        .sort_values("_time"))
solar_sorted = df_solar.sort_values("_time").copy()
df = pd.merge_asof(
    solar_sorted, weer, on="_time",
    direction="nearest", tolerance=pd.Timedelta("2min"),
    suffixes=("", "_weer")
)
df["delta_T"] = df["Pt100"] - df["Temperature-PVMET500"]
df["P_mpp"] = df["impp"] * df["vmpp"]
print(f"Gekoppelde rijen: {len(df):,}")
print(f"Met geldige omgevingstemperatuur: {df['Temperature-PVMET500'].notna().sum():,}")
display(df.groupby("label")[["Pt100", "delta_T", "P_mpp"]].count())

## 4. Weersomstandigheden en paneeltemperatuur

Vergelijk de paneeltemperatuur met de omgevingstemperatuur. Bij PV8 zijn eerder opvallend lage temperaturen waargenomen; behandel dit voorlopig als mogelijke sensor- of registratieafwijking, niet als bewezen fysisch effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for paneel, g in df.groupby("label"):
    d = g.dropna(subset=["_time", "Pt100"])
    axes[0].scatter(d["_time"], d["Pt100"], s=5, alpha=.45, label=paneel.replace("PvDevice-", "PV"))
axes[0].plot(weer["_time"], weer["Temperature-PVMET500"], color="black", linewidth=1.3, label="Omgeving")
axes[0].set_title("Paneel- en omgevingstemperatuur")
axes[0].set_ylabel("Temperatuur (°C)")
axes[0].legend(ncol=3, fontsize=8)
axes[0].tick_params(axis="x", rotation=25)

data_delta = [g["delta_T"].dropna() for _, g in sorted(df.groupby("label"))]
namen = [p.replace("PvDevice-", "PV") for p, _ in sorted(df.groupby("label"))]
axes[1].boxplot(data_delta, tick_labels=namen, showfliers=False)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Paneel minus omgeving")
axes[1].set_ylabel("ΔT (°C)")
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()
plt.show()
display(df.groupby("label")["delta_T"].agg(["count", "median", "min", "max"]).round(2))

## 5. Temperatuur en elektrische prestaties

Het gemeten vermogen bij het maximum power point wordt benaderd als **Pₘₚₚ = Iₘₚₚ × Vₘₚₚ**. Correlatie betekent geen causaliteit: instraling beïnvloedt zowel temperatuur als vermogen. Vergelijkingen tussen panelen zijn bovendien voorlopig door mogelijke gedeelde elektrische meetkanalen.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for paneel, g in df.groupby("label"):
    naam = paneel.replace("PvDevice-", "PV")
    for ax, y, titel, eenheid in [
        (axes[0], "voc", "Openklemspanning", "V"),
        (axes[1], "ff", "Fill factor", "ff"),
        (axes[2], "P_mpp", "Berekend maximaal vermogen", "W")
    ]:
        d = g.dropna(subset=["Pt100", y])
        ax.scatter(d["Pt100"], d[y], s=10, alpha=.45, label=naam)
        ax.set_title(titel)
        ax.set_xlabel("Paneeltemperatuur (°C)")
        ax.set_ylabel(eenheid)
        ax.grid(alpha=.2)
axes[2].legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

for y in ["voc", "ff", "P_mpp"]:
    print(f"\nCorrelatie Pt100 versus {y} (beschrijvend):")
    print(df.groupby("label").apply(
        lambda g: g[["Pt100", y]].dropna().corr().iloc[0, 1]
        if len(g[["Pt100", y]].dropna()) >= 3 else np.nan,
        include_groups=False
    ).round(3))

## 6. CMP-10: verkennende analyse

**De betekenis, eenheid en kalibratie van CMP-10 zijn niet bevestigd.** De naam doet vermoeden dat het een instralingssensor betreft. We onderzoeken uitsluitend het tijdspatroon en de samenhang met stroom; we noemen de waarden nadrukkelijk nog geen W/m².

In [ ]:
if "CMP-10" in df.columns:
    cmp = (df[["_time", "CMP-10"]].dropna()
           .groupby("_time", as_index=False)["CMP-10"].median()
           .sort_values("_time"))
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(cmp["_time"], cmp["CMP-10"], linewidth=1, color="#c68c20")
    ax.set_title("CMP-10 (betekenis en eenheid onbekend)")
    ax.set_xlabel("Datum (UTC)")
    ax.set_ylabel("CMP-10 (onbekende eenheid)")
    ax.grid(alpha=.2)
    plt.tight_layout()
    plt.show()
    display(cmp["CMP-10"].describe().round(2))
else:
    print("Geen CMP-10-kolom aanwezig.")

## 7. Vergelijkbaarheid en datakwaliteit

We controleren (a) het aantal geldige waarden per paneel en (b) of stroommetingen van verschillende panelen op dezelfde tijdstippen identiek zijn. Grote overeenkomsten kunnen duiden op gedeelde registratie; dat moet bij de opdrachtgever worden nagevraagd.

In [ ]:
display(df.groupby("label")[["Pt100", "impp", "isc", "vmpp", "voc", "P_mpp"]].count())
stroom = df.pivot_table(index="_time", columns="label", values="impp", aggfunc="first")
records = []
from itertools import combinations
for a, b in combinations(stroom.columns, 2):
    paar = stroom[[a, b]].dropna()
    if len(paar):
        gelijk = np.isclose(paar[a], paar[b], rtol=0, atol=1e-9)
        records.append({"Paneel A": a, "Paneel B": b,
                        "Gelijktijdig": len(paar), "Identiek": int(gelijk.sum()),
                        "Overeenkomst (%)": round(100*gelijk.mean(), 1)})
vergelijking = pd.DataFrame(records)
if not vergelijking.empty:
    display(vergelijking.sort_values("Overeenkomst (%)", ascending=False).head(15))

## 8. Conclusies en vervolgstappen

Vul na het uitvoeren van de cellen de onderstaande bevindingen aan. Scheid **observaties** van **verklaringen**.

**Voorlopige aandachtspunten**
- PV8 heeft eerder afwijkende negatieve paneeltemperaturen laten zien; controleer de sensorpositie en kalibratie.
- Niet alle elektrische meetwaarden zijn op alle tijdstippen beschikbaar.
- Sommige panelen vertonen sterk overlappende of identieke elektrische reeksen; controleer de meetarchitectuur.
- `CMP-10` is nog niet fysisch te interpreteren zonder metadata.
- Voor degradatieonderzoek zijn gegevens over meerdere jaren en paneelspecificaties nodig.

**Nog in te vullen na uitvoering**
1. Hoe groot zijn de temperatuurverschillen tussen paneel en omgeving?
2. Welke samenhang zien we tussen temperatuur en spanning, fill factor of vermogen?
3. Welke vergelijkingen zijn betrouwbaar gezien ontbrekende en identieke waarden?
4. Welke extra gegevens zijn nodig (instraling, wind, paneeltype, hellingshoek, kalibratie)?

**Publicatie:** neem alleen geaggregeerde en door de opdrachtgever goedgekeurde resultaten over op de GitHub Pages-site.